# 🚀 Hackathon Cloud GPU Training (Google Colab)
Use this notebook on Google Colab's **free 15GB T4 GPU** to train 7B/8B parameter models if you need higher VRAM than your local 4GB GPU, then export the LoRA weights back to your local hackathon repo!

In [ ]:
!nvidia-smi
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = 'unsloth/Llama-3.2-3B-Instruct'
print(f'Loading model {model_id} with 4-bit quantization...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Export to zip and copy into hackathon-repo/ai_ml/checkpoints/